# DermaVision AI: train on Kaggle (no downloads, no uploads of data)
This notebook contains all the code. Before running:
1. Right sidebar → **Settings → Accelerator → GPU T4 x2 (or P100)**.
2. Right sidebar → **Settings → Internet → On** (needed once to fetch the pretrained EfficientNet weights). Kaggle may ask you to verify your phone number for GPU and Internet.
3. **Add Input** (top right) → search `skin-cancer-mnist-ham10000` → click the **+** next to the dataset by *kmader*.

Then run the cells in order. Educational use only: HAM10000 is licensed CC BY-NC 4.0.

## 1. Check GPU and dataset

In [ ]:
!nvidia-smi | head -n 12
!ls /kaggle/input/skin-cancer-mnist-ham10000

## 2. Write the project code files (just run these 4 cells)

In [ ]:
%%writefile common.py
"""Shared pieces for training/evaluation. The backend (backend/app/services/classifier.py) must preprocess
images exactly like `eval_transform` below, or accuracy will silently drop."""
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms as T
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

# HAM10000 diagnosis codes, alphabetical. The checkpoint stores this list so the backend never has to guess.
CLASS_NAMES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]
IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]


def build_model(num_classes: int = len(CLASS_NAMES), pretrained: bool = True) -> torch.nn.Module:
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    model.classifier[1] = torch.nn.Linear(model.classifier[1].in_features, num_classes)
    return model


# Whole image is squashed to a square first (same as the backend), then augmented.
eval_transform = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), T.Normalize(MEAN, STD)])
train_transform = T.Compose(
    [
        T.Resize((IMG_SIZE, IMG_SIZE)),
        T.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0), ratio=(0.9, 1.1)),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),  # lesions have no "up", so flips and full rotations are safe
        T.RandomRotation(180),
        T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ]
)


class SkinDataset(Dataset):
    def __init__(self, csv_path, transform):
        self.df = pd.read_csv(csv_path)
        self.transform = transform
        self.targets = [CLASS_NAMES.index(d) for d in self.df["dx"]]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        img = Image.open(self.df.iloc[i]["image_path"]).convert("RGB")
        return self.transform(img), self.targets[i]

In [ ]:
%%writefile prepare_data.py
"""Build train/val/test CSVs from HAM10000, splitting BY LESION so the same lesion never appears in two splits.

HAM10000 has ~10,015 images but only ~7,470 unique lesions: many lesions were photographed several times.
A naive random split leaks near-duplicate images into the test set and inflates accuracy.

Usage:  python prepare_data.py --data-dir data/HAM10000
"""
import argparse
from pathlib import Path

import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True, help="Folder with HAM10000_metadata.csv and the image folders")
    ap.add_argument("--out-dir", default=None, help="Where to write the split CSVs (default: <data-dir>/splits)")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    root = Path(args.data_dir)
    meta = pd.read_csv(next(root.rglob("HAM10000_metadata.csv")))
    paths = {p.stem: str(p.resolve()) for p in root.rglob("*.jpg")}
    meta["image_path"] = meta["image_id"].map(paths)
    missing = meta["image_path"].isna().sum()
    if missing:
        raise SystemExit(f"{missing} images listed in the metadata were not found under {root}")

    # 10 folds grouped by lesion: fold 0 = test (~10%), fold 1 = validation (~10%), rest = train (~80%).
    folds = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=args.seed)
    meta["fold"] = -1
    for k, (_, idx) in enumerate(folds.split(meta, meta["dx"], groups=meta["lesion_id"])):
        meta.loc[meta.index[idx], "fold"] = k

    splits = {"test": meta[meta.fold == 0], "val": meta[meta.fold == 1], "train": meta[meta.fold >= 2]}
    ids = {k: set(v["lesion_id"]) for k, v in splits.items()}
    assert not (ids["train"] & ids["val"] or ids["train"] & ids["test"] or ids["val"] & ids["test"]), "lesion leakage!"

    out = Path(args.out_dir) if args.out_dir else root / "splits"
    out.mkdir(parents=True, exist_ok=True)
    cols = ["image_path", "dx", "lesion_id"]
    for name, df in splits.items():
        df[cols].to_csv(out / f"{name}.csv", index=False)
        print(f"{name:5s} {len(df):5d} images, {df.lesion_id.nunique():5d} lesions  ", df.dx.value_counts().to_dict())
    print(f"Wrote splits to {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile train.py
"""Fine-tune EfficientNet-B0 on HAM10000.

    python train.py --data-dir data/HAM10000/splits

Phase 1 (warm-up): backbone frozen, only the new classifier head trains.
Phase 2 (fine-tune): everything trains at a lower learning rate with cosine decay.
Afterwards a temperature is fitted on the validation set so the confidence % shown in the app is calibrated.
"""
import argparse
import copy
from collections import Counter
from pathlib import Path

import numpy as np
import torch
from sklearn.metrics import balanced_accuracy_score, f1_score, recall_score
from torch import nn
from torch.utils.data import DataLoader, Subset

from common import CLASS_NAMES, IMG_SIZE, SkinDataset, build_model, eval_transform, train_transform

MEL = CLASS_NAMES.index("mel")


@torch.no_grad()
def predict_logits(model, loader, device):
    model.eval()
    logits, labels = [], []
    for x, y in loader:
        logits.append(model(x.to(device)).float().cpu())
        labels.append(y)
    return torch.cat(logits), torch.cat(labels)


def metrics(logits, labels):
    pred = logits.argmax(1).numpy()
    y = labels.numpy()
    return {
        "accuracy": float((pred == y).mean()),
        "balanced_accuracy": float(balanced_accuracy_score(y, pred)),
        "macro_f1": float(f1_score(y, pred, average="macro", zero_division=0)),
        "melanoma_recall": float(recall_score(y, pred, labels=[MEL], average="macro", zero_division=0)),
    }


def fit_temperature(logits, labels):
    """Single-parameter calibration: softmax(logits / T). T > 1 softens over-confident predictions."""
    log_t = torch.zeros(1, requires_grad=True)
    opt = torch.optim.LBFGS([log_t], lr=0.1, max_iter=100)
    nll = nn.CrossEntropyLoss()

    def closure():
        opt.zero_grad()
        loss = nll(logits / log_t.exp(), labels)
        loss.backward()
        return loss

    opt.step(closure)
    return float(log_t.exp())


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True, help="Folder containing train.csv / val.csv (from prepare_data.py)")
    ap.add_argument("--out", default="../backend/weights/dermavision_effnet_b0.pt")
    ap.add_argument("--epochs", type=int, default=25)
    ap.add_argument("--warmup-epochs", type=int, default=3)
    ap.add_argument("--batch-size", type=int, default=32)
    ap.add_argument("--head-lr", type=float, default=1e-3)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--patience", type=int, default=6, help="stop if val macro-F1 does not improve for this many epochs")
    ap.add_argument("--workers", type=int, default=4)
    ap.add_argument("--no-pretrained", action="store_true", help="skip ImageNet weights (offline smoke tests only)")
    ap.add_argument("--max-samples", type=int, default=0, help="use only N samples per split (smoke tests)")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    torch.manual_seed(args.seed)
    np.random.seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
    print("device:", device)

    d = Path(args.data_dir)
    train_ds, val_ds = SkinDataset(d / "train.csv", train_transform), SkinDataset(d / "val.csv", eval_transform)
    if args.max_samples:
        train_ds = Subset(train_ds, range(args.max_samples))
        val_ds = Subset(val_ds, range(args.max_samples))
        train_targets = [train_ds.dataset.targets[i] for i in train_ds.indices]
    else:
        train_targets = train_ds.targets
    pin = device == "cuda"
    train_dl = DataLoader(train_ds, args.batch_size, shuffle=True, num_workers=args.workers, pin_memory=pin, drop_last=True)
    val_dl = DataLoader(val_ds, args.batch_size, shuffle=False, num_workers=args.workers, pin_memory=pin)

    # HAM10000 is ~67% nevus and ~1% dermatofibroma. Sqrt-inverse-frequency weights help rare classes
    # (melanoma especially) without the instability of full inverse-frequency weighting.
    counts = Counter(train_targets)
    w = torch.tensor([1.0 / np.sqrt(max(counts.get(i, 1), 1)) for i in range(len(CLASS_NAMES))], dtype=torch.float32)
    w = (w / w.mean()).to(device)
    criterion = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    print("class counts:", {CLASS_NAMES[i]: counts.get(i, 0) for i in range(len(CLASS_NAMES))})

    model = build_model(pretrained=not args.no_pretrained).to(device)
    amp = device == "cuda"
    scaler = torch.amp.GradScaler(enabled=amp)

    def make_opt(params, lr, epochs_left):
        opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
        return opt, torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs_left, 1))

    best, best_state, best_epoch = -1.0, None, 0
    for epoch in range(args.epochs):
        if epoch == 0 and args.warmup_epochs > 0:
            for p in model.features.parameters():
                p.requires_grad = False
            opt, sched = make_opt(model.classifier.parameters(), args.head_lr, args.warmup_epochs)
            print(f"-- phase 1: training head only for {args.warmup_epochs} epochs")
        if epoch == args.warmup_epochs:
            for p in model.parameters():
                p.requires_grad = True
            opt, sched = make_opt(model.parameters(), args.lr, args.epochs - epoch)
            print("-- phase 2: fine-tuning all layers")
        if epoch == 0 and args.warmup_epochs == 0:
            opt, sched = make_opt(model.parameters(), args.lr, args.epochs)

        model.train()
        total = 0.0
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device, enabled=amp):
                loss = criterion(model(x), y)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            total += loss.item()
        sched.step()

        logits, labels = predict_logits(model, val_dl, device)
        m = metrics(logits, labels)
        print(f"epoch {epoch + 1:2d}/{args.epochs}  loss {total / max(len(train_dl), 1):.3f}  "
              + "  ".join(f"{k} {v:.3f}" for k, v in m.items()))
        if m["macro_f1"] > best:
            best, best_epoch, best_state, best_metrics = m["macro_f1"], epoch, copy.deepcopy(model.state_dict()), m
        elif epoch - best_epoch >= args.patience:
            print("early stopping")
            break

    model.load_state_dict(best_state)
    logits, labels = predict_logits(model, val_dl, device)
    temperature = fit_temperature(logits, labels)
    print(f"best epoch {best_epoch + 1}, val macro-F1 {best:.3f}, fitted temperature {temperature:.2f}")

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "state_dict": {k: v.cpu() for k, v in best_state.items()},
            "class_names": CLASS_NAMES,
            "img_size": IMG_SIZE,
            "temperature": temperature,
            "val_metrics": best_metrics,
        },
        out,
    )
    print("saved", out.resolve())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile evaluate.py
"""Evaluate a trained checkpoint on the held-out TEST split.

    python evaluate.py --data-dir data/HAM10000/splits --ckpt ../backend/weights/dermavision_effnet_b0.pt

Look at per-class recall (sensitivity), especially melanoma, not just overall accuracy: with ~67% nevi in the data,
a model that always says "nevus" already scores ~67% accuracy.
"""
import argparse
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader

from common import SkinDataset, build_model, eval_transform


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data-dir", required=True)
    ap.add_argument("--ckpt", default="../backend/weights/dermavision_effnet_b0.pt")
    ap.add_argument("--batch-size", type=int, default=64)
    ap.add_argument("--workers", type=int, default=4)
    args = ap.parse_args()

    device = "cuda" if torch.cuda.is_available() else "cpu"
    ckpt = torch.load(args.ckpt, map_location="cpu", weights_only=True)
    names, temp = ckpt["class_names"], ckpt.get("temperature", 1.0)
    model = build_model(len(names), pretrained=False)
    model.load_state_dict(ckpt["state_dict"])
    model.to(device).eval()

    ds = SkinDataset(Path(args.data_dir) / "test.csv", eval_transform)
    dl = DataLoader(ds, args.batch_size, num_workers=args.workers)
    probs, labels = [], []
    with torch.no_grad():
        for x, y in dl:
            probs.append(torch.softmax(model(x.to(device)).cpu() / temp, 1))
            labels.append(y)
    probs, y = torch.cat(probs).numpy(), torch.cat(labels).numpy()
    pred = probs.argmax(1)

    print(classification_report(y, pred, target_names=names, digits=3, zero_division=0))
    cm = confusion_matrix(y, pred, labels=range(len(names)))

    # Expected calibration error: gap between stated confidence and actual accuracy.
    conf, correct = probs.max(1), (pred == y).astype(float)
    bins, ece = np.linspace(0, 1, 11), 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.any():
            ece += m.mean() * abs(correct[m].mean() - conf[m].mean())
    print(f"accuracy {correct.mean():.3f}   expected calibration error {ece:.3f}")

    mel = names.index("mel")
    mel_true = y == mel
    print(f"melanoma sensitivity {(pred[mel_true] == mel).mean():.3f}   "
          f"specificity {(pred[~mel_true] != mel).mean():.3f}")

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(cm / cm.sum(1, keepdims=True).clip(1), cmap="Blues")
    ax.set_xticks(range(len(names)), names, rotation=45)
    ax.set_yticks(range(len(names)), names)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=8)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    fig.tight_layout()
    out = Path(args.ckpt).with_name("confusion_matrix.png")
    fig.savefig(out, dpi=150)
    print("saved", out)


if __name__ == "__main__":
    main()

## 3. Split the data by lesion

In [ ]:
!python prepare_data.py --data-dir /kaggle/input/skin-cancer-mnist-ham10000 --out-dir /kaggle/working/splits

## 4. Smoke test (about 1 to 2 minutes). Low scores are normal here.

In [ ]:
!python train.py --data-dir /kaggle/working/splits --epochs 1 --warmup-epochs 1 --max-samples 256 --workers 2 --out /kaggle/working/smoke.pt

## 5. Full training (about 30 to 60 minutes)
Keep this browser tab open.

In [ ]:
!python train.py --data-dir /kaggle/working/splits --epochs 25 --workers 2 --out /kaggle/working/dermavision_effnet_b0.pt

## 6. Evaluate on the untouched test split
Look at recall for `mel` (melanoma).

In [ ]:
!python evaluate.py --data-dir /kaggle/working/splits --ckpt /kaggle/working/dermavision_effnet_b0.pt --workers 2
from IPython.display import Image
Image('/kaggle/working/confusion_matrix.png')

## 7. Download the weights
In the right sidebar open **Data → Output → /kaggle/working**, click the three dots next to `dermavision_effnet_b0.pt`, and choose **Download**.
Then copy it into `DermaVisionAI/backend/weights/`.